In [7]:
import pandas as pd
data = pd.read_csv(r"C:\Users\DELL\Downloads\telecom_customer_churn.csv")
data.head()

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [31]:
model_data = data[['Customer Status', 'Phone Service', 'Online Security', 'Online Backup','Device Protection Plan','Streaming TV','Streaming Music', 'Premium Tech Support']]
model_data.head()

,Customer Status,Phone Service,Online Security,Online Backup,Device Protection Plan,Streaming TV,Streaming Music,Premium Tech Support
0,Stayed,Yes,No,Yes,No,Yes,No,Yes
1,Stayed,Yes,No,No,No,No,Yes,No
2,Churned,Yes,No,No,Yes,No,No,No
3,Churned,Yes,No,Yes,Yes,Yes,No,No
4,Churned,Yes,No,No,No,Yes,No,Yes


In [33]:
yes_no_cols = ['Phone Service', 'Online Security', 'Online Backup',
               'Device Protection Plan', 'Premium Tech Support',
               'Streaming TV', 'Streaming Music']

model_data[yes_no_cols] = model_data[yes_no_cols].apply(lambda col: col.map({'Yes': 1, 'No': 0}))

In [35]:
model_data.head()

,Customer Status,Phone Service,Online Security,Online Backup,Device Protection Plan,Streaming TV,Streaming Music,Premium Tech Support
0,Stayed,1,0.0,1.0,0.0,1.0,0.0,1.0
1,Stayed,1,0.0,0.0,0.0,0.0,1.0,0.0
2,Churned,1,0.0,0.0,1.0,0.0,0.0,0.0
3,Churned,1,0.0,1.0,1.0,1.0,0.0,0.0
4,Churned,1,0.0,0.0,0.0,1.0,0.0,1.0


In [37]:
# Drop any rows with missing values first (logistic regression can't handle NaN)
model_data = model_data.dropna()

X = model_data.drop(columns=['Customer Status'])
y = model_data['Customer Status']

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Coefficients tell you direction and strength of impact
coefficients = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print(coefficients)

# Odds ratios are usually easier to interpret than raw coefficients
odds_ratios = np.exp(coefficients)
print(odds_ratios)

Premium Tech Support     -0.446236
Online Security          -0.391227
Streaming TV              0.358837
Phone Service             0.264216
Streaming Music           0.256283
Online Backup            -0.128386
Device Protection Plan    0.023599
dtype: float64
Premium Tech Support      0.640033
Online Security           0.676226
Streaming TV              1.431663
Phone Service             1.302409
Streaming Music           1.292118
Online Backup             0.879514
Device Protection Plan    1.023880
dtype: float64


In [49]:
import statsmodels.api as sm

X_sm = sm.add_constant(X)  # adds an intercept term
logit_model = sm.Logit(y, X_sm)
result = logit_model.fit()

print(result.summary())

ImportError: cannot import name '_lazywhere' from 'scipy._lib._util' (C:\Users\DELL\anaconda3\Lib\site-packages\scipy\_lib\_util.py)